In [3]:
import os
import sys
sys.path.append('..')
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import dataframe_image as dfi
import ipywidgets as widgets
from IPython.display import display, clear_output
from sqlalchemy import create_engine, text
from docxtpl import DocxTemplate, InlineImage
from docx.shared import Inches
from datetime import datetime

# --- 1. CONFIGURACIÓN DE RUTAS Y BASE DE DATOS ---
ruta_base = r"G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA"
ruta_plantilla = r"G:\OneDrive - Ingenio Azucarero Guabira S.A\_DATOS_PYTHON\templates"
ruta_shp_lotes = r"G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SHP_LOTES_COSECHA_FRENTE_1\PROPIEDADES.shp"

# El shapefile de propiedades está en UTM zona 20S (WGS84 / UTM 20S = EPSG:32720)
EPSG_LOTES = 32720

from config import POSTGRES_UTEA
POSTGRES_UTEA['DATABASE'] = 'utea_precision'

def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# Carga de ambas bases de datos CSV
df_ha = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_HA_COSECHADA.csv"), encoding="utf-8")
df_detalles = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_DETALLE_LOTES.csv"), encoding="utf-8")

# Carga del shapefile de propiedades/lotes (capa de contorno para el plano de TCH)
try:
    gdf_lotes_shp = gpd.read_file(ruta_shp_lotes)
    if gdf_lotes_shp.crs is None:
        gdf_lotes_shp.set_crs(epsg=EPSG_LOTES, inplace=True)
    elif gdf_lotes_shp.crs.to_epsg() != EPSG_LOTES:
        gdf_lotes_shp = gdf_lotes_shp.to_crs(epsg=EPSG_LOTES)
except Exception as e:
    print(f"⚠️ No se pudo cargar el shapefile de lotes ({ruta_shp_lotes}): {e}")
    gdf_lotes_shp = gpd.GeoDataFrame()

# =====================================================================
# 🔥 ¡AÑADE ESTAS LÍNEAS AQUÍ ABAJO PARA CORREGIR EL ERROR!
# =====================================================================
# Aseguramos que las columnas de cálculos de df_ha sean números limpios
for col in ['Area Cosechada', 'Tn Real']:
    if col in df_ha.columns:
        # Reemplaza comas por puntos (si vienen en formato latino) y convierte a float
        df_ha[col] = df_ha[col].astype(str).str.replace(',', '.').str.strip()
        df_ha[col] = pd.to_numeric(df_ha[col], errors='coerce').fillna(0.0)

# Aseguramos que las columnas de df_detalles sean números limpios
for col in ['Area Cosecha', 'Tn Real', 'TCH Real', 'POND ME', 'POND PCF', 'POND FIBRA', 'POND PUREZA']:
    if col in df_detalles.columns:
        df_detalles[col] = df_detalles[col].astype(str).str.replace(',', '.').str.strip()
        df_detalles[col] = pd.to_numeric(df_detalles[col], errors='coerce').fillna(0.0)
# =====================================================================

meses_espanol = {
    1: 'enero', 2: 'febrero', 3: 'marzo', 4: 'abril', 5: 'mayo', 6: 'junio',
    7: 'julio', 8: 'agosto', 9: 'septiembre', 10: 'octubre', 11: 'noviembre', 12: 'diciembre'
}

# Paleta de colores actualizada: Rojo, Naranja, Amarillo, Verde Claro y Verde Intenso
colores_tch = {
    '< 40 Tn/Ha': '#d7191c',       # Rojo Intenso (Rendimiento muy bajo)
    '40 a 60 Tn/Ha': '#fdae61',    # Naranja (Rendimiento regular)
    '60 a 80 Tn/Ha': '#ffffbf',    # Amarillo (Rendimiento bueno / promedio)
    '80 a 100 Tn/Ha': '#a6d96a',   # Verde Claro (Rendimiento muy bueno)
    '> 100 Tn/Ha': '#1a9641'       # Verde Intenso (Rendimiento óptimo / sobresaliente)
}

# --- 2. FUNCIÓN PARA EXTRACTO GEOGRÁFICO DE LA PROPIEDAD COMPLETA ---
def obtener_telemetria_propiedad(propiedad):
    """Descarga los recorridos de todos los lotes pertenecientes a una misma propiedad."""
    engine = obtener_engine()
    query = text("""
        SELECT id, vryldrcane, distance, swathwidth, distancia_real, geom 
        FROM datos_cosecha.recorrido_cosechadora 
        WHERE propiedad = :prop
    """)
    try:
        gdf = gpd.read_postgis(query, engine, geom_col='geom', params={"prop": propiedad})
        return gdf
    except Exception as e:
        print(f"⚠️ Nota: No se pudo extraer mapa desde Postgres para esta propiedad: {e}")
        return gpd.GeoDataFrame()

def cargar_capa_lote_propiedad(propiedad, crs_destino=None):
    """Filtra el contorno de la propiedad seleccionada en el shapefile de lotes
    (campo 'unidad_02') y lo reproyecta al CRS de la telemetría si es necesario.
    Devuelve un GeoDataFrame vacío si no se encuentra la propiedad o falta el campo."""
    if gdf_lotes_shp.empty or 'unidad_02' not in gdf_lotes_shp.columns:
        return gpd.GeoDataFrame()

    lotes_prop = gdf_lotes_shp[gdf_lotes_shp['unidad_02'] == propiedad].copy()
    if lotes_prop.empty:
        return lotes_prop

    if crs_destino is not None and lotes_prop.crs is not None and lotes_prop.crs != crs_destino:
        lotes_prop = lotes_prop.to_crs(crs_destino)

    return lotes_prop

def generar_plano_y_tabla_tch(gdf, path_img, path_tabla, propiedad):
    """Genera el plano general (filtrado) y el cuadro estadístico (sin filtrar) con colores coordinados."""
    gdf = gdf.copy()
    
    # NUEVA CATEGORIZACIÓN EXTENDIDA (Aplica para mapa y estadísticas)
    gdf['categoria'] = pd.cut(
        gdf['vryldrcane'], 
        bins=[-1, 40, 60, 80, 100, 9999],
        labels=['< 40 Tn/Ha', '40 a 60 Tn/Ha', '60 a 80 Tn/Ha', '80 a 100 Tn/Ha', '> 100 Tn/Ha'], 
        include_lowest=True
    )
    
    # --- 1. GENERAR PLANO VISUAL (Filtrado por distancia_real < 3) ---
    col_filtro = 'distancia_real' if 'distancia_real' in gdf.columns else 'distance'
    gdf_plano = gdf[gdf[col_filtro] < 3].copy()

    # Contorno de la propiedad (shapefile PROPIEDADES.shp, filtrado por unidad_02),
    # alineado al mismo CRS que la telemetría para que ambas capas calcen
    lotes_prop_shp = cargar_capa_lote_propiedad(propiedad, crs_destino=gdf.crs)
    
    # --- ANTES: figsize=(10, 5) ---
    # --- AHORA: figsize=(14, 14) ---
    fig, ax = plt.subplots(figsize=(14, 14))

    # Límites combinando el recorrido de la cosechadora y el contorno del lote,
    # para que ambas capas queden completas dentro del plano
    minx, miny, maxx, maxy = gdf_plano.total_bounds
    if not lotes_prop_shp.empty:
        lminx, lminy, lmaxx, lmaxy = lotes_prop_shp.total_bounds
        minx, miny = min(minx, lminx), min(miny, lminy)
        maxx, maxy = max(maxx, lmaxx), max(maxy, lmaxy)
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    # Márgenes independientes en X e Y (sin forzar un rango cuadrado):
    # así, al recortar el espacio en blanco con bbox_inches='tight', la imagen
    # final se ajusta a la forma real del recorrido y lo aprovecha por completo.
    rango_x = (maxx - minx) * 1.1
    rango_y = (maxy - miny) * 1.1
    
    # Renderizar tramos por categoría de color con el grosor definido (linewidth)
    for cat, data in gdf_plano.groupby('categoria', observed=False):
        data.plot(ax=ax, color=colores_tch.get(cat, '#7f8c8d'), linewidth=0.4, label=str(cat))

    # Capa de contorno de la propiedad: sin relleno, contorno rojo, con etiqueta del lote
    if not lotes_prop_shp.empty:
        lotes_prop_shp.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=1.5, zorder=5)
        if 'unidad_05' in lotes_prop_shp.columns:
            lotes_prop_shp.apply(
                lambda x: ax.annotate(
                    text=str(x['unidad_05']),
                    xy=x.geometry.representative_point().coords[0],
                    ha='center', va='center', color='black', fontsize=9, weight='bold',
                    bbox=dict(facecolor=(1, 1, 1, 0.5), edgecolor='none', pad=1),
                    zorder=6
                ), axis=1
            )
        
    ax.set_xlim(cx - rango_x / 2, cx + rango_x / 2)
    ax.set_ylim(cy - rango_y / 2, cy + rango_y / 2)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # --- ANTES: dpi=100 ---
    # --- AHORA: Subimos a dpi=300 para calidad de impresión profesional (Alta Definición) ---
    # bbox_inches='tight' recorta el margen blanco sobrante para que los recorridos
    # ocupen toda la imagen, sin estirar ni distorsionar la forma real (aspect='equal' se mantiene)
    plt.savefig(path_img, dpi=300, bbox_inches='tight')
    plt.close()

    # --- 2. GENERAR CUADRO ESTADÍSTICO (Sin Filtrar - Totalidad de Datos Reales) ---
    gdf['area_ha'] = (gdf['distance'] * gdf['swathwidth']) / 10000
    
    resumen = gdf.groupby('categoria', observed=False).agg(
        Metros=('distance', 'sum'),
        Hectareas=('area_ha', 'sum')
    ).reset_index()
    
    total_m = resumen['Metros'].sum() if resumen['Metros'].sum() > 0 else 1
    resumen['Distribución (%)'] = (resumen['Metros'] / total_m) * 100
    
    col_name = 'TCH (Sensor)'
    resumen.columns = [col_name, 'Longitud Total', 'Superficie', 'Distribución (%)']

    # Fila de TOTAL: suma de Longitud Total y Superficie, y 100% de Distribución
    fila_total = pd.DataFrame({
        col_name: ['TOTAL'],
        'Longitud Total': [resumen['Longitud Total'].sum()],
        'Superficie': [resumen['Superficie'].sum()],
        'Distribución (%)': [resumen['Distribución (%)'].sum()]
    })
    resumen = pd.concat([resumen, fila_total], ignore_index=True)
    
    # Función de pintado dinámico de celdas según categoría (Sincronizado)
    def aplicar_color_categoria(row):
        categoria_fila = row[col_name]
        estilos = [''] * len(row)

        # Fila de TOTAL: fondo gris y negrita en toda la fila, sin color de categoría
        if categoria_fila == 'TOTAL':
            estilos = ['background-color: #f2f2f2; font-weight: bold;'] * len(row)
            return estilos

        color_hex = colores_tch.get(categoria_fila, '#ffffff')
        idx_distribucion = row.index.get_loc('Distribución (%)')
        
        # El texto va en negro si el fondo es amarillo o naranja/verde claro para una lectura nítida
        text_color = 'black' if color_hex in ['#fdae61', '#ffffbf', '#a6d96a'] else 'white'
        estilos[idx_distribucion] = f'background-color: {color_hex}; color: {text_color}; font-weight: bold;'
        return estilos

    style = resumen.style.apply(aplicar_color_categoria, axis=1).hide(axis='index')
    style = style.format({
        'Longitud Total': '{:,.1f} m',
        'Superficie': '{:,.2f} ha',
        'Distribución (%)': '{:,.1f} %'
    })
    
    dfi.export(style, path_tabla, table_conversion='chrome')

def color_por_estado(estado):
    """Devuelve el color de fondo (hex, sin #) para la celda Estado según su texto."""
    e = str(estado).strip().lower()
    if 'cosechado' in e:
        return 'A9D08E'   # verde claro
    if 'iniciado' in e:
        return 'FFE699'   # amarillo claro
    return 'auto'         # sin color de fondo para otros estados

# --- 3. FUNCIÓN DE RENDERIZADO PRINCIPAL ---
def generar_reporte_servicio_completo(propiedad, fecha_control):
    try:
        plantilla_path = os.path.join(ruta_plantilla, "tpl_inf_cosecha_servicio_F1.docx")
        doc = DocxTemplate(plantilla_path)
        
        anio_sel = fecha_control.year
        mes_nombre_sel = meses_espanol[fecha_control.month]
        semana_sel = fecha_control.isocalendar()[1]
        
        # --- BLOQUE 1: CÁLCULOS MATEMÁTICOS (df_ha) ---
        df_prop = df_ha[(df_ha['Propiedad'] == propiedad) & (df_ha['Año'] == anio_sel)]
        sem_area = df_prop[(df_prop['Mes'].str.lower() == mes_nombre_sel) & (df_prop['Semana'] == semana_sel)]['Area Cosechada'].sum()
        sem_tn = df_prop[(df_prop['Mes'].str.lower() == mes_nombre_sel) & (df_prop['Semana'] == semana_sel)]['Tn Real'].sum()
        sem_tch = (sem_tn / sem_area) if sem_area > 0 else 0.0
        
        mes_area = df_prop[df_prop['Mes'].str.lower() == mes_nombre_sel]['Area Cosechada'].sum()
        mes_tn = df_prop[df_prop['Mes'].str.lower() == mes_nombre_sel]['Tn Real'].sum()
        mes_tch = (mes_tn / mes_area) if mes_area > 0 else 0.0
        
        total_area = df_prop['Area Cosechada'].sum()
        total_tn = df_prop['Tn Real'].sum()
        total_tch = (total_tn / total_area) if total_area > 0 else 0.0
        
        # --- BLOQUE 2: PROCESAMIENTO DE LAS DOS TABLAS DE LOTES ---
        
        df_lotes_prop = df_detalles[(df_detalles['Propiedad'] == propiedad) & (df_detalles['Area Cosecha'] > 0.01)].copy()
        
        df_lotes_prop['Area Cosecha'] = df_lotes_prop['Area Cosecha'].fillna(0.0)
        df_lotes_prop['Tn Real'] = df_lotes_prop['Tn Real'].fillna(0.0)
        df_lotes_prop['TCH Real'] = df_lotes_prop['TCH Real'].fillna(0.0)
        df_lotes_prop['Variedad'] = df_lotes_prop['Variedad'].fillna("S/V")
        df_lotes_prop['Soca'] = df_lotes_prop['Soca'].fillna(0)
        df_lotes_prop['POND ME'] = df_lotes_prop['POND ME'].fillna(0.0)
        df_lotes_prop['POND PCF'] = df_lotes_prop['POND PCF'].fillna(0.0)
        df_lotes_prop['POND FIBRA'] = df_lotes_prop['POND FIBRA'].fillna(0.0)
        df_lotes_prop['POND PUREZA'] = df_lotes_prop['POND PUREZA'].fillna(0.0)

        # --- Totales para la fila TOTAL de la tabla "RESUMEN DE AVANCE POR LOTE" ---
        tot_area_lotes = df_lotes_prop['Area Cosecha'].sum()
        tot_tn_lotes = df_lotes_prop['Tn Real'].sum()
        tot_tch_lotes = (tot_tn_lotes / tot_area_lotes) if tot_area_lotes > 0 else 0.0

        # --- Totales (promedio ponderado por Tn Real) para la tabla "RESUMEN DE CALIDAD POR LOTE" ---
        if tot_tn_lotes > 0:
            tot_me = (df_lotes_prop['POND ME'] * df_lotes_prop['Tn Real']).sum() / tot_tn_lotes
            tot_pcf = (df_lotes_prop['POND PCF'] * df_lotes_prop['Tn Real']).sum() / tot_tn_lotes
            tot_fibra = (df_lotes_prop['POND FIBRA'] * df_lotes_prop['Tn Real']).sum() / tot_tn_lotes
            tot_pureza = (df_lotes_prop['POND PUREZA'] * df_lotes_prop['Tn Real']).sum() / tot_tn_lotes
        else:
            tot_me = tot_pcf = tot_fibra = tot_pureza = 0.0

        lista_lotes_jinja = []
        for _, fila in df_lotes_prop.iterrows():
            avance_original = fila['Avance']
            if pd.isna(avance_original):
                avance_final = "0 %"
            else:
                try:
                    avance_limpio = str(avance_original).replace('%', '').replace(',', '.').strip()
                    avance_num = float(avance_limpio)
                    if avance_num > 100.0: avance_num = 100.0
                    avance_final = f"{int(round(avance_num))} %"
                except:
                    avance_final = str(avance_original)
            
            lista_lotes_jinja.append({
                "lote": str(fila['Lote']), "estado": str(fila['Estado Cosecha']), "avance": avance_final,
                "estado_color": color_por_estado(fila['Estado Cosecha']),
                "variedad": str(fila['Variedad']), "soca": str(int(fila['Soca'])),
                "area": f"{fila['Area Cosecha']:,.2f}", "tn": f"{fila['Tn Real']:,.2f}", "tch": f"{fila['TCH Real']:,.2f}",
                "me": f"{fila['POND ME']:,.2f} %", "pcf": f"{fila['POND PCF']:,.2f}",
                "fibra": f"{fila['POND FIBRA']:,.2f} %", "pureza": f"{fila['POND PUREZA']:,.2f} %"
            })
            
        # --- BLOQUE 3: CONSTRUCCIÓN DEL MAPA Y LA TABLA GENERAL (PostgreSQL) ---
        img_temp_mapa = "tmp_tch_general.png"
        img_temp_tabla = "tmp_tch_general_tb.png"
        gdf_geo = obtener_telemetria_propiedad(propiedad)
        tiene_geo = not gdf_geo.empty
        
        if tiene_geo:
            generar_plano_y_tabla_tch(gdf_geo, img_temp_mapa, img_temp_tabla, propiedad)
            
        # --- BLOQUE 4: CONTEXTO INTEGRADO PARA JINJA ---
        contexto_r = {
            "fecha": fecha_control.strftime("%d/%m/%Y"),
            "propiedad": propiedad,
            "num_semana": semana_sel,
            "nombre_mes": mes_nombre_sel.capitalize(),
            
            "sem_area": f"{sem_area:,.2f}", "sem_tn": f"{sem_tn:,.2f}", "sem_tch": f"{sem_tch:,.2f}",
            "mes_area": f"{mes_area:,.2f}", "mes_tn": f"{mes_tn:,.2f}", "mes_tch": f"{mes_tch:,.2f}",
            "total_area": f"{total_area:,.2f}", "total_tn": f"{total_tn:,.2f}", "total_tch": f"{total_tch:,.2f}",
            
            "lotes": lista_lotes_jinja,

            # Totales de la tabla "RESUMEN DE AVANCE POR LOTE"
            "tot_area_lotes": f"{tot_area_lotes:,.2f}",
            "tot_tn_lotes": f"{tot_tn_lotes:,.2f}",
            "tot_tch_lotes": f"{tot_tch_lotes:,.2f}",

            # Promedios (ponderados por área) de la tabla "RESUMEN DE CALIDAD POR LOTE"
            "tot_me": f"{tot_me:,.2f} %",
            "tot_pcf": f"{tot_pcf:,.2f}",
            "tot_fibra": f"{tot_fibra:,.2f} %",
            "tot_pureza": f"{tot_pureza:,.2f} %",
            
            # Inyección de Imagen del Plano e Imagen del Cuadro Estadístico
            "img_plano_tch_general": InlineImage(doc, img_temp_mapa, width=Inches(6.0)) if tiene_geo else "[Sin datos geoespaciales]",
            "img_tabla_tch_general": InlineImage(doc, img_temp_tabla, width=Inches(4.0)) if tiene_geo else ""
        }
        
        doc.render({"r": contexto_r})
        nombre_archivo = f"Reporte_Servicio_Completo_{propiedad.replace(' ', '_')}_Semana_{semana_sel}.docx"
        ruta_salida = os.path.join(ruta_base, nombre_archivo)
        doc.save(ruta_salida)
        
        # Limpieza de archivos temporales
        for f in [img_temp_mapa, img_temp_tabla]:
            if os.path.exists(f): os.remove(f)
        
        return f"✅ ¡Reporte consolidado con éxito!\n📂 Archivo: {ruta_salida}\n🌍 Incluye: Tablas operativas, de calidad, plano y cuadro estadístico general de TCH de la propiedad."
        
    except Exception as e:
        return f"❌ Error al procesar el reporte: {str(e)}"

# --- 5. INTERFAZ GRÁFICA INTERACTIVA ---
combo_propiedad = widgets.Dropdown(options=sorted(df_ha['Propiedad'].unique()), description='Propiedad:')
control_fecha = widgets.DatePicker(description='Fecha Consulta:', value=datetime.now().date())
btn_generar = widgets.Button(description='Generar Reporte Completo', button_style='success', icon='map')
output_data = widgets.Output()

def controlar_interfaz(*args):
    with output_data:
        clear_output()
        if combo_propiedad.value and control_fecha.value:
            print(f"Propiedad: {combo_propiedad.value} | Fecha seleccionada: {control_fecha.value.strftime('%d/%m/%Y')}")
            cant_lotes = len(df_detalles[df_detalles['Propiedad'] == combo_propiedad.value])
            print(f"📋 Se procesará la cartografía unificada y el cuadro estadístico global para {cant_lotes} lotes.")
            display(btn_generar)

def al_hacer_clic_generar(b):
    with output_data:
        print("\n📥 Conectando a PostGIS, extrayendo telemetría general y renderizando tablas con Jinja...")
        mensaje = generar_reporte_servicio_completo(combo_propiedad.value, control_fecha.value)
        print(mensaje)

combo_propiedad.observe(controlar_interfaz, 'value')
control_fecha.observe(controlar_interfaz, 'value')
btn_generar.on_click(al_hacer_clic_generar)

controlar_interfaz()
display(widgets.HBox([combo_propiedad, control_fecha]), output_data)

Output()